In [ ]:
# run first
%load_ext autoreload
%autoreload 2

#### Imports

In [ ]:
import os
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt

from aicspylibczi import CziFile
from PIL import Image, ImageDraw, ImageFont
from pipeline.methods import to_directory, read_czi, read_yaml, read_beads, read_h5, \
     read_shape, read_kernel, to_mip, to_shift, configure_plots, to_shifted, to_ellipsoid, \
     make_color, to_fmt, segmentation, detection, finding, deskewing, deconvolution, colors

### Commands

#### 0. Find beads (all CZIs)
``` bash
python beads.py -b beads.txt -c config.yaml -m finding
```

#### 1. Segment nuclei (single CZI)
``` bash
python segmentation.py -f file.czi -c config.yaml
```

#### 2. Track nuclei
``` bash
python tracking.py -f file.czi -c config.yaml -mode nuclei
```

#### 3. Detect features (within nuclei)
``` bash
python detection.py -f file.czi -c config.yaml
```

#### 4. Track features (within nuclei)
``` bash
python tracking.py -f file.czi -c config.yaml -m features
```

#### 5. Fitting chromatic aberrations (and bead intensities)
``` bash
python beads.py -f file.czi -b beads.txt -c config.yaml -m fitting
```

#### 6. Measure feature tracks
``` bash
python measuring.py -f file.czi -b beads.txt -c config.yaml
```

#### Configure

In [ ]:
# inputs
file = "../../../projects/synEP/movies/20250219/G7B8G2_GSK_500uMdTAG13added2hrBefore_30ms_0_3umIntervals_15by650_30sFrameRate-02.czi"
beads = "beads.txt"
psf = "PSFs.h5"
directory = to_directory(file)
config = "config.yaml"

# read CZI and config
czi = CziFile(file)
print(czi.size)
c = read_yaml(config)

# path to steps/nuclei
frames_path = os.path.join(directory, "frames")
nuclei_path = os.path.join(directory, "nuclei")

# get PSF parameters
lasers = c["general"]["lasers"]
kernel = read_kernel(psf, lasers)
shape = read_shape(psf, lasers)

# set GPU and plot formatting
device = 0
configure_plots()

#### Read

In [ ]:
# read a frame
frame = 0
stack = read_czi(file, frame)

### Deskewing

In [ ]:
# deskew all channels
with cp.cuda.Device(device):
    deskew = deskewing(stack, **c["deskewing"])

In [ ]:
# make MIP and show
chnl = 0
mip = to_mip(deskew[chnl], **c["movies"]["channels"][chnl], axis=0)
display(Image.fromarray(np.transpose(mip, axes=(1,0,2))))

In [ ]:
# make MIP and show
chnl = 0
mip = to_mip(deskew[chnl], **c["movies"]["channels"][chnl], axis=1)
display(Image.fromarray(np.flipud(mip)))

In [ ]:
# make MIP and show
chnl = 0
mip = to_mip(deskew[chnl], **c["movies"]["channels"][chnl], axis=2)
display(Image.fromarray(np.flipud(mip)))

### Deconvolution

In [ ]:
# deconvolve all channels
with cp.cuda.Device(device):
    deconvolve = deconvolution(deskew, kernel, **c["deconvolution"])

In [ ]:
# make MIP and show
chnl = 0
mip = to_mip(deconvolve[chnl], **c["movies"]["channels"][chnl], axis=0)
display(Image.fromarray(np.transpose(mip, axes=(1,0,2))))

In [ ]:
# make MIP and show
chnl = 0
mip = to_mip(deconvolve[chnl], **c["movies"]["channels"][chnl], axis=1)
display(Image.fromarray(np.flipud(mip)))

In [ ]:
# make MIP and show
chnl = 0
mip = to_mip(deconvolve[chnl], **c["movies"]["channels"][chnl], axis=2)
display(Image.fromarray(np.flipud(mip)))

### Segmentation

In [ ]:
# segment nuclei
with cp.cuda.Device(device):
    labels, info = segmentation(deconvolve, **c["segmentation"])

#### Show nuclei labels

In [ ]:
# make MIP
chnl = c["segmentation"]["channel"]
mip = to_mip(deconvolve[chnl], **c["movies"]["channels"][chnl], gray=True, axis=0)
mip = np.clip(mip, 0, 200)  # clip for adding labels

# add color to MIP
mip_labels = np.max(labels, axis=0)
for i in range(len(info)):
    mip[mip_labels == i] += (55*make_color()).astype(np.uint8)
mip = Image.fromarray(np.transpose(mip, axes=(1,0,2)))

# write labels
draw = ImageDraw.Draw(mip)
for i in range(len(info)):
    x, y, label = info[i,2], info[i,3], i
    draw.text(xy=(x-20,y-15),text=to_fmt(label),
              font=ImageFont.truetype(font="Arial", size=25), fill=(0,0,0))
display(mip)

#### Subset frame

In [ ]:
nucleus = 23

# get subset
l, u = info[nucleus,4:7].astype(int), info[nucleus,7:10].astype(int)
dcv = np.copy(deconvolve[:,l[0]:u[0],l[1]:u[1],l[2]:u[2]])
dsk = np.copy(deskew[:,l[0]:u[0],l[1]:u[1],l[2]:u[2]])
mask = labels[l[0]:u[0],l[1]:u[1],l[2]:u[2]] == nucleus

#### Show nucleus

In [ ]:
# make merged MIP of channels
mips = []
for chnl in range(len(dcv)):
    for axis in [0,1]:
        mip = to_mip(dcv[chnl], mask=mask, axis=axis, **c["movies"]["channels"][chnl])
        mips.append(np.transpose(mip, axes=(1,0,2)))  # transpose
mips = np.concatenate(mips, axis=1)
display(Image.fromarray(mips))

#### Registration of channels

In [ ]:
# channels
ref, trg = 0, 1
with cp.cuda.Device(device):
    # run registration
    shift = to_shift(dsk[ref], dsk[trg])
    print(f"z: {shift[0]:.2f} y: {shift[1]:.2f} x: {shift[2]:.2f}")

In [ ]:
# make MIPs of channels
ref_mip = to_mip(dsk[ref], **c["movies"]["channels"][ref], axis=0)
trg_mip = to_mip(to_shifted(dsk[trg], shift), **c["movies"]["channels"][trg], axis=0)
display(Image.fromarray(np.transpose(np.maximum(ref_mip, trg_mip), axes=(1,0,2))))

### Detection

#### Detect in channels

In [ ]:
# detect features
with cp.cuda.Device(device):
    features = detection(dcv, dsk, mask, shape, **c["detection"])

#### Show detections

In [ ]:
# make merged MIP of channels
mips = []
for chnl in range(len(dcv)):
    for axis in [0,1]:
        mip = to_mip(dcv[chnl], mask=mask, axis=axis, **c["movies"]["channels"][chnl])
        mips.append(np.transpose(mip, axes=(1,0,2)))  # transpose
mips = Image.fromarray(np.concatenate(mips, axis=1))

# show detections
draw, radius = ImageDraw.Draw(mips), 8
offset = np.sum(dcv.shape[1:3])
for chnl in c["detection"]["channels"]:
    pts = features[chnl]
    for j in range(len(pts)):
        color = "yellow" if pts[j,-1] else "black"
        x, y = pts[j,4] + chnl*offset, pts[j,5]
        draw.ellipse((x-radius, y-radius, x+radius, y+radius), outline=color, width=2)
        x, y = pts[j,3] + chnl*offset + dcv.shape[2], pts[j,5]
        draw.ellipse((x-radius, y-radius, x+radius, y+radius), outline=color, width=2)
display(mips)

#### Show detected within nucleus

In [ ]:
# read nucleus track
nucleus = 47
nucleus_track = np.loadtxt(os.path.join(nuclei_path, f"{to_fmt(nucleus)}", "info.csv"),
                           delimiter=",", skiprows=1)

In [ ]:
for i in range(10):
    # get frame/label
    frame, label = int(nucleus_track[i,0]), int(nucleus_track[i,1])
    frame_path = os.path.join(frames_path, f"{to_fmt(frame)}")

    # read signal/mask and detect features
    lbl = f"{to_fmt(label)}"
    mask = read_h5(os.path.join(frame_path, "masks.h5"), lbl)
    dcv = read_h5(os.path.join(frame_path, "deconvolves.h5"), lbl)
    with cp.cuda.Device(device):
        features = detection(dcv, dcv, mask, shape, **c["detection"])

    # make merged MIP of channels
    mips = []
    for chnl in range(len(dcv)):
        for axis in [0,1]:
            mip = to_mip(dcv[chnl], mask=mask, axis=axis, **c["movies"]["channels"][chnl])
            mips.append(np.transpose(mip, axes=(1,0,2)))  # transpose
    mips = Image.fromarray(np.concatenate(mips, axis=1))

    # show detections
    draw, radius = ImageDraw.Draw(mips), 8
    offset = np.sum(dcv.shape[1:3])
    for chnl in c["detection"]["channels"]:
        pts = features[chnl]
        for j in range(len(pts)):
            color= "yellow" if pts[j,-1] else "black"
            x, y = pts[j,4] + chnl*offset, pts[j,5]
            draw.ellipse((x-radius, y-radius, x+radius, y+radius), outline=color, width=2)
            x, y = pts[j,3] + chnl*offset + dcv.shape[2], pts[j,5]
            draw.ellipse((x-radius, y-radius, x+radius, y+radius), outline=color, width=2)
    display(mips)

### Features

#### Show tracks

In [ ]:
# iterate through tracks
path = os.path.join(nuclei_path, f"{to_fmt(nucleus)}", "tracks")
for item in os.listdir(path):
    if item[:-4].isnumeric():
        # make figure and read track from CSV
        fig, axis = plt.subplots(figsize=(5,2), dpi=200)
        axis.set_xlabel("Time (min)")
        axis.spines["right"].set_visible(True)
        track = np.loadtxt(os.path.join(path, item), delimiter=",", skiprows=1)

        # plot intensities
        intensities = track[:,25]
        axis.plot(track[:,1]/60, intensities, color=colors["cyan"])
        axis.set_ylabel("Intensity (au)")
        axis.tick_params(axis="y", labelcolor=colors["cyan"])

        # plot distances
        twin = axis.twinx()
        distances = track[:,23]
        distances[distances > 600] = np.nan
        twin.plot(track[:,1]/60, distances, color=colors["orange"])
        twin.set_ylabel("Distance (nm)")
        twin.tick_params(axis="y", labelcolor=colors["orange"])
        plt.show()

# Beads

### Read bead movie

In [ ]:
# read bead paths and get random
paths = read_beads(beads)
path = paths[np.random.randint(0, len(paths))]

# read frame
frame = 0
raw = read_czi(path, frame)  # reading deskew
print(path)

In [ ]:
# deskew all channels
with cp.cuda.Device(device):
    deskew = deskewing(raw, **c["deskewing"])

### Detect in channels

In [ ]:
# detect beads in channel
with cp.cuda.Device(device):
    positions, _ = finding(deskew, **c["beads"]["finding"])

In [ ]:
# make MIP and show
chnl = 0
mip = to_mip(deskew[chnl]/3, **c["movies"]["channels"][chnl], axis=0)
mip = Image.fromarray(np.transpose(mip, axes=(1,0,2)))

# show beads
draw, radius = ImageDraw.Draw(mip), 12
for j in range(len(positions[chnl])):
    y, x = positions[chnl][j,1], positions[chnl][j,2]
    draw.ellipse((y-radius, x-radius, y+radius, x+radius), outline="yellow", width=3)
display(mip)

### Show PSF volume and mask

In [ ]:
chnl = 0

# get kernel/shape volumes
scale = 2.0
volumes = [kernel[chnl], to_ellipsoid(scale**2 * shape[chnl])]
fig, axes = plt.subplots(2, 3, figsize=(15,10))

# show all axes
k = kernel.shape[1]
for col in range(3):
    for row in range(len(volumes)):
        # make MIP
        mip = to_mip(volumes[row], axis=col,
                     color=c["movies"]["channels"][chnl]["color"])
        # show image
        h, w = mip.shape[:2]
        axes[row,col].set(xlim=(0, k), ylim=(0, k))
        axes[row,col].imshow(mip, extent=((k-w)/2, (k+w)/2, (k-h)/2, (k+h)/2))
        axes[row,col].axis("off")

        # write dimension labels
        dims = ["Z", "Y", "X"]
        dims.pop(col)
        if row == 0:
            axes[row,col].text(0.5, 1.25, dims[0], fontsize=20, color="w")
            axes[row,col].text(1.25, 0.5, dims[1], fontsize=20, color="w")
plt.tight_layout()
plt.show()